# Stage 1 · Data Pipeline — Step 3: Generate Client Dataset Splits

This notebook generates **IID** and **Non-IID** versions of the hospital datasets required for Federated Learning experiments.

## Why Non-IID Matters in Federated Learning

In real-world healthcare, hospitals serve **different patient populations**. A hospital in an urban area with an older, more sedentary population may see significantly **higher diabetes prevalence** than a rural hospital serving younger, more active patients. This creates **statistical heterogeneity (Non-IID data)** — the most challenging and realistic scenario for Federated Learning.

| Scenario | Description | Challenge |
|----------|-------------|----------|
| **IID** | Both hospitals see the same class distribution (~14% diabetic) | Low — baseline |
| **Non-IID** | NY: ~22% diabetic prevalence \| TX: ~8% | High — simulates real heterogeneity |

> **Research Motivation**: FedProx proximal term is specifically designed to handle Non-IID drift. This notebook creates the data needed to validate that claim empirically.

In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

# Load the base hospital datasets (produced by previous pipeline steps)
ny_df = pd.read_csv('../data/processed/Hospital_NY.csv')
tx_df = pd.read_csv('../data/processed/Hospital_TX.csv')

print(f'NY  | Total: {len(ny_df):,} | Diabetic: {ny_df.Diabetes_binary.mean():.1%}')
print(f'TX  | Total: {len(tx_df):,} | Diabetic: {tx_df.Diabetes_binary.mean():.1%}')

OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)


NY  | Total: 126,840 | Diabetic: 13.9%
TX  | Total: 126,840 | Diabetic: 13.9%


## Part A: IID Split

**IID (Independent and Identically Distributed)** represents a perfect scenario where the federated data partition matches the overall population distribution. We apply a **stratified 50/50 split** of the merged dataset so both hospitals see the same class balance. This serves as the controlled baseline for comparison.

In [2]:
from sklearn.model_selection import train_test_split

# Merge, shuffle, then stratified 50/50 split
merged = pd.concat([ny_df, tx_df], axis=0).reset_index(drop=True)
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

ny_iid, tx_iid = train_test_split(
    merged,
    test_size=0.5,
    random_state=42,
    stratify=merged['Diabetes_binary']
)

ny_iid.to_csv(f'{OUTPUT_DIR}/Hospital_NY_IID.csv', index=False)
tx_iid.to_csv(f'{OUTPUT_DIR}/Hospital_TX_IID.csv', index=False)

print('IID Split complete:')
print(f'  NY_IID  | Rows: {len(ny_iid):,} | Diabetic: {ny_iid.Diabetes_binary.mean():.1%}')
print(f'  TX_IID  | Rows: {len(tx_iid):,} | Diabetic: {tx_iid.Diabetes_binary.mean():.1%}')


IID Split complete:
  NY_IID  | Rows: 126,840 | Diabetic: 13.9%
  TX_IID  | Rows: 126,840 | Diabetic: 13.9%


## Part B: Non-IID Split

We simulate **hospital-specific population heterogeneity** using controlled biased sampling:

- **Hospital NY (High Prevalence)**: Target ~22% diabetic — simulates an urban hospital serving an older, more sedentary population.
- **Hospital TX (Low Prevalence)**: Target ~8% diabetic — simulates a younger, more active patient cohort.

**Implementation**: We separate the merged dataset by label, then draw samples at different positive/negative ratios for each hospital. This ensures the total dataset size remains comparable to the IID split while creating a meaningful distribution shift — the key condition FedProx must overcome.

> **IEEE Note**: The Non-IID split is a controlled simulation, not a ground-truth demographic split. It is clearly documented as such in the paper methodology.

In [3]:
def create_noniid_split(df: pd.DataFrame, target_pos_rate: float,
                        n_total: int, seed: int = 42) -> pd.DataFrame:
    """
    Create a dataset subset with a controlled positive-class prevalence.

    Args:
        df            : Source DataFrame with 'Diabetes_binary' column.
        target_pos_rate : Desired diabetic prevalence (0.0 – 1.0).
        n_total       : Total number of rows in the output.
        seed          : Random seed for reproducibility.

    Returns:
        DataFrame with the requested class distribution.
    """
    rng = np.random.default_rng(seed)

    positives = df[df['Diabetes_binary'] == 1]
    negatives = df[df['Diabetes_binary'] == 0]

    n_pos = int(n_total * target_pos_rate)
    n_neg = n_total - n_pos

    # Sample with replacement if needed to hit exact counts
    sampled_pos = positives.sample(n=n_pos, replace=(n_pos > len(positives)),
                                   random_state=seed)
    sampled_neg = negatives.sample(n=n_neg, replace=(n_neg > len(negatives)),
                                   random_state=seed)

    result = pd.concat([sampled_pos, sampled_neg]).sample(frac=1,
                        random_state=seed).reset_index(drop=True)
    return result


half = len(merged) // 2

# NY: high prevalence hospital (~22% diabetic)
ny_noniid = create_noniid_split(merged, target_pos_rate=0.22, n_total=half, seed=42)

# TX: low prevalence hospital (~8% diabetic)
tx_noniid = create_noniid_split(merged, target_pos_rate=0.08, n_total=half, seed=42)

ny_noniid.to_csv(f'{OUTPUT_DIR}/Hospital_NY_NONIID.csv', index=False)
tx_noniid.to_csv(f'{OUTPUT_DIR}/Hospital_TX_NONIID.csv', index=False)

print('Non-IID Split complete:')
print(f'  NY_NONIID | Rows: {len(ny_noniid):,} | Diabetic: {ny_noniid.Diabetes_binary.mean():.1%}')
print(f'  TX_NONIID | Rows: {len(tx_noniid):,} | Diabetic: {tx_noniid.Diabetes_binary.mean():.1%}')


Non-IID Split complete:
  NY_NONIID | Rows: 126,840 | Diabetic: 22.0%
  TX_NONIID | Rows: 126,840 | Diabetic: 8.0%


## Summary & Validation

The table below confirms each output file exists and has the correct class distribution before these datasets are used for federated training experiments.

In [4]:
summary = []
files = {
    'Hospital_NY_IID.csv':    'IID — NY (balanced)',
    'Hospital_TX_IID.csv':    'IID — TX (balanced)',
    'Hospital_NY_NONIID.csv': 'Non-IID — NY (high prevalence)',
    'Hospital_TX_NONIID.csv': 'Non-IID — TX (low prevalence)',
}

for fname, desc in files.items():
    path = f'{OUTPUT_DIR}/{fname}'
    assert os.path.exists(path), f'MISSING: {path}'
    df_ = pd.read_csv(path)
    summary.append({
        'File': fname,
        'Description': desc,
        'Rows': f'{len(df_):,}',
        'Diabetic %': f'{df_.Diabetes_binary.mean():.1%}',
        'Healthy %': f'{(1 - df_.Diabetes_binary.mean()):.1%}',
    })

print(pd.DataFrame(summary).to_string(index=False))
print('\n✅ All 4 split files validated successfully.')


                  File                    Description    Rows Diabetic % Healthy %
   Hospital_NY_IID.csv            IID — NY (balanced) 126,840      13.9%     86.1%
   Hospital_TX_IID.csv            IID — TX (balanced) 126,840      13.9%     86.1%
Hospital_NY_NONIID.csv Non-IID — NY (high prevalence) 126,840      22.0%     78.0%
Hospital_TX_NONIID.csv  Non-IID — TX (low prevalence) 126,840       8.0%     92.0%

✅ All 4 split files validated successfully.
